# 02 - eda (sample)

visualize sample data: class distribution, waveforms, mel spectrograms.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import librosa
import librosa.display
from pathlib import Path

SAMPLES = Path("../../data/samples")
df = pd.read_csv(SAMPLES / "sample_labels.csv")
PROCESSED = Path("../../data/processed")
sns.set_theme(style='whitegrid')
plt.rcParams['figure.dpi'] = 120
%matplotlib inline
print(f"{len(df)} sample files")

In [ ]:
fig, axes = plt.subplots(1,2,figsize=(12,4))
for ax,(ch,sub) in zip(axes, df.groupby('channel')):
    c = sub['emotion'].value_counts().sort_index()
    ax.bar(c.index, c.values, color=plt.cm.Set2(np.linspace(0,1,len(c))))
    ax.set_title(f'{ch} ({len(sub)} files)')
    ax.tick_params(axis='x', rotation=45)
plt.tight_layout(); plt.show()

In [ ]:
sample = df.sample(20, random_state=42)
durs = []
for fp in sample['filepath']:
    y,sr = librosa.load(fp, duration=10)
    durs.append(len(y)/sr)
durs = np.array(durs)
print(f"duration: min={durs.min():.2f}s max={durs.max():.2f}s mean={durs.mean():.2f}s")

In [ ]:
fig, axes = plt.subplots(4,2,figsize=(12,8))
axes = axes.flatten()
for i,(_,row) in enumerate(df.groupby('emotion').first().iterrows()):
    y,sr = librosa.load(row['filepath'])
    axes[i].plot(np.arange(len(y))/sr, y, linewidth=0.3)
    axes[i].set_title(f"{row['emotion']}")
plt.tight_layout(); plt.show()

In [ ]:
fig, axes = plt.subplots(4,2,figsize=(12,10))
axes = axes.flatten()
for i,(_,row) in enumerate(df.groupby('emotion').first().iterrows()):
    y,sr = librosa.load(row['filepath'])
    mel = librosa.feature.melspectrogram(y=y, sr=sr, n_mels=128)
    logmel = librosa.power_to_db(mel, ref=np.max)
    librosa.display.specshow(logmel, x_axis='time', y_axis='mel', sr=sr, ax=axes[i])
    axes[i].set_title(f"{row['emotion']}")
plt.tight_layout(); plt.show()

eda done on sample data. shows expected class balance, distinct waveform patterns per emotion.